# COS40007 - Structural Defect Detection (Colab All-in-One)

Train and evaluate **YOLOv11n**, **YOLOv8n** and **SSDLite** on the balanced
5-class defect dataset, end to end, in a single notebook.

## How to use on Google Colab
1. **Runtime -> Change runtime type -> T4 GPU** (or any GPU).
2. Upload **`defect_dataset.zip`** to your Drive at `MyDrive/COS40007/defect_dataset.zip`
   (drag-and-drop in the Drive web page - reliable for large files).
3. **Runtime -> Run all.** Trained weights + charts are zipped back to Drive at the end.

Classes: `cracks, spalling, corrosion, potholes, paint_degradation`.

> This notebook is self-contained: it needs only the data zip. The SSDLite model
> definition is inlined, so no extra project files are required on Colab.


## Part 0 - Environment Setup

In [ ]:
# 0.1 Install deps (Colab has torch but not ultralytics/torchmetrics)
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

if IN_COLAB:
    pip_install('ultralytics', 'torchmetrics')
print('Running on Colab:', IN_COLAB)

In [ ]:
# 0.2 Mount Drive, copy the data zip to LOCAL disk, and unzip.
# Training off mounted Drive is very slow (per-file latency) - always copy local.
import os, zipfile, shutil
from pathlib import Path

# >>> EDIT THIS if you put the zip somewhere else on Drive <<<
DRIVE_ZIP_PATH = '/content/drive/MyDrive/COS40007/defect_dataset.zip'

WORK_DIR = Path('/content/defect_project') if IN_COLAB else Path('..').resolve()

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    if not (WORK_DIR / 'data').exists():
        assert os.path.exists(DRIVE_ZIP_PATH), (
            'Zip not found at %s - upload defect_dataset.zip to Drive first.' % DRIVE_ZIP_PATH)
        local_zip = WORK_DIR / 'defect_dataset.zip'
        print('Copying zip from Drive to local disk...')
        shutil.copy(DRIVE_ZIP_PATH, local_zip)
        print('Unzipping...')
        with zipfile.ZipFile(local_zip) as z:
            z.extractall(WORK_DIR)
        print('Done.')
    else:
        print('data/ already present - skipping unzip.')
else:
    print('Local run - using existing project data/.')

In [ ]:
# 0.3 Paths, classes, device, and data.yaml (rewritten with the correct abs path)
PROJECT_ROOT = WORK_DIR
DATA_DIR = PROJECT_ROOT / 'data'
RUNS_DIR = PROJECT_ROOT / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ['cracks', 'spalling', 'corrosion', 'potholes', 'paint_degradation']
CLASS_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
NUM_CLASSES = len(CLASSES) + 1  # +1 background (torchvision convention)
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

def write_data_yaml():
    (DATA_DIR / 'data.yaml').write_text(
        'path: %s\n'
        'train: images/train\nval: images/val\ntest: images/test\n'
        'nc: %d\nnames: %s\n' % (DATA_DIR.as_posix(), len(CLASSES), CLASSES)
    )

assert DATA_DIR.exists(), 'data/ not found at %s' % DATA_DIR
write_data_yaml()
DATA_YAML = str(DATA_DIR / 'data.yaml')
print('data.yaml ->', DATA_YAML)

In [ ]:
# 0.4 Hyperparameters - auto-scaled to the GPU.
# A 4GB laptop GPU needs small batches + workers=0; a Colab T4/L4 (>=15GB) can do more.
GPU_MEM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0.0
BIG_GPU = GPU_MEM_GB >= 12

YOLO_EPOCHS  = 100
TORCH_EPOCHS = 30
YOLO_BATCH   = 16 if BIG_GPU else 8
TORCH_BATCH  = 8  if BIG_GPU else 2
WORKERS      = 2 if (IN_COLAB or os.name != 'nt') else 0   # Windows must use 0
LR           = 1e-4
IMG_SIZE     = 640
USE_AMP      = torch.cuda.is_available()

print('GPU mem: %.1f GB | big_gpu=%s' % (GPU_MEM_GB, BIG_GPU))
print('YOLO: batch=%d epochs=%d | SSDLite: batch=%d epochs=%d | workers=%d | amp=%s'
      % (YOLO_BATCH, YOLO_EPOCHS, TORCH_BATCH, TORCH_EPOCHS, WORKERS, USE_AMP))

## Part 1 - Dataset Overview (EDA)

Confirms the data loaded correctly and documents class balance for the report.

In [ ]:
# 1.1 Per-split image counts and per-class bounding-box counts
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def list_images(split):
    d = DATA_DIR / 'images' / split
    return [p for p in d.iterdir() if p.suffix.lower() in IMAGE_EXTS] if d.exists() else []

def count_bbox_per_class(split):
    counts = {c: 0 for c in CLASSES}
    for lbl in (DATA_DIR / 'labels' / split).glob('*.txt'):
        for line in lbl.read_text().splitlines():
            parts = line.split()
            if parts:
                cid = int(parts[0])
                if 0 <= cid < len(CLASSES):
                    counts[CLASSES[cid]] += 1
    return counts

rows = [{'split': s, 'images': len(list_images(s)), **count_bbox_per_class(s)}
        for s in ('train', 'val', 'test')]
stats_df = pd.DataFrame(rows).set_index('split')
print(stats_df.to_string())

In [ ]:
# 1.2 Class-distribution chart - stacked by split (shows the per-class balance)
split_colors = {'train': '#34495e', 'val': '#7f8c8d', 'test': '#bdc3c7'}
tallies = {s: count_bbox_per_class(s) for s in ('train', 'val', 'test')}

fig, ax = plt.subplots(figsize=(9.5, 4.5))
bottom = [0] * len(CLASSES)
for s in ('train', 'val', 'test'):
    vals = [tallies[s][c] for c in CLASSES]
    ax.bar(CLASSES, vals, bottom=bottom, label=s, color=split_colors[s],
           edgecolor='white', width=0.6)
    bottom = [b + v for b, v in zip(bottom, vals)]
for i, total in enumerate(bottom):
    ax.text(i, total + 15, str(total), ha='center', va='bottom', fontsize=9, fontweight='bold')
ratio = (max(bottom) / min(bottom)) if min(bottom) else 0
ax.set_ylabel('Number of bounding boxes')
ax.set_title('Class Distribution by Split (balance ratio: %.2fx)' % ratio)
ax.legend(title='split')
plt.tight_layout()
plt.savefig(RUNS_DIR / 'class_distribution.png', dpi=150)
plt.show()

In [ ]:
# 1.3 One labelled sample image per class
def find_img(stem):
    for ext in IMAGE_EXTS:
        p = DATA_DIR / 'images' / 'train' / (stem + ext)
        if p.exists():
            return p
    return None

def draw_bboxes(ax, img, label_path):
    w, h = img.size
    ax.imshow(img)
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
            x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
            ax.add_patch(patches.Rectangle((x1, y1), bw * w, bh * h, lw=2,
                         edgecolor=CLASS_COLORS[cls], facecolor='none'))
    ax.axis('off')

lbl_dir = DATA_DIR / 'labels' / 'train'
by_class = {c: [] for c in range(len(CLASSES))}
for lbl in lbl_dir.glob('*.txt'):
    t = lbl.read_text().strip()
    if t:
        by_class[int(t.split()[0])].append(lbl.stem)

fig, axes = plt.subplots(1, len(CLASSES), figsize=(20, 4))
for ci, name in enumerate(CLASSES):
    ax = axes[ci]
    if by_class[ci]:
        stem = by_class[ci][0]
        p = find_img(stem)
        if p:
            draw_bboxes(ax, Image.open(p).convert('RGB'), lbl_dir / (stem + '.txt'))
    ax.set_title(name)
plt.suptitle('Sample Training Images with Ground-Truth Boxes')
plt.tight_layout()
plt.savefig(RUNS_DIR / 'sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 2 - Training

Trains three detectors with transfer learning, each auto-versioned into `runs/`.
- **YOLOv11n** and **YOLOv8n** (Ultralytics, anchor-free) - COCO-pretrained.
- **SSDLite320-MobileNetV3** (torchvision, anchor-based) - ImageNet backbone,
  two-phase: freeze backbone for 5 epochs, then unfreeze at a 10x lower LR.

In [ ]:
# 2.0 Shared run logger + version helper
import time, csv

def log_run(name, epochs, batch, map50, map50_95, precision, recall,
            train_min, params_m, weights, notes=''):
    log_path = RUNS_DIR / 'run_log.csv'
    new = not log_path.exists()
    with open(log_path, 'a', newline='') as f:
        w = csv.writer(f)
        if new:
            w.writerow(['run_id', 'model', 'version', 'epochs', 'batch', 'imgsz',
                        'lr', 'map50', 'map50_95', 'precision', 'recall',
                        'train_time_min', 'params_M', 'weights', 'notes'])
        run_id = '%s_%d' % (name, int(time.time()))
        w.writerow([run_id, name, '', epochs, batch, IMG_SIZE, LR,
                    round(map50, 4), round(map50_95, 4),
                    round(precision, 4), round(recall, 4),
                    round(train_min, 1), round(params_m, 2), weights, notes])
    print('Logged run:', run_id)

def next_version(subdir):
    v = 1
    while (RUNS_DIR / subdir / ('v%d' % v)).exists():
        v += 1
    return v

### 2.1 YOLOv11n

In [ ]:
from ultralytics import YOLO

def train_yolo(weights_name, subdir, label):
    ver = next_version(subdir)
    print('%s -> runs/%s/v%d' % (label, subdir, ver))
    model = YOLO(weights_name)  # auto-downloads COCO weights on Colab
    t0 = time.time()
    model.train(data=DATA_YAML, epochs=YOLO_EPOCHS, imgsz=IMG_SIZE, batch=YOLO_BATCH,
                lr0=LR, device=0 if torch.cuda.is_available() else 'cpu',
                project=str(RUNS_DIR / subdir), name='v%d' % ver, exist_ok=False,
                save=True, amp=USE_AMP, workers=WORKERS, verbose=True)
    mins = (time.time() - t0) / 60
    val = model.val(data=DATA_YAML)
    w = str(RUNS_DIR / subdir / ('v%d' % ver) / 'weights' / 'best.pt')
    params = sum(p.numel() for p in model.model.parameters()) / 1e6
    print('%s: mAP@0.5=%.4f mAP@0.5:0.95=%.4f (%.0f min)'
          % (label, val.box.map50, val.box.map, mins))
    log_run(label, YOLO_EPOCHS, YOLO_BATCH, float(val.box.map50), float(val.box.map),
            float(val.box.mp), float(val.box.mr), mins, params, w)
    return model

yolo11 = train_yolo('yolo11n.pt', 'yolo11n', 'YOLOv11n')

### 2.2 YOLOv8n

In [ ]:
yolo8 = train_yolo('yolov8n.pt', 'yolov8n', 'YOLOv8n')

### 2.3 SSDLite320-MobileNetV3 (model + dataset inlined)

In [ ]:
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models import MobileNet_V3_Large_Weights
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm

# --- SSDLite builder (inlined from models/ssdlite_detector.py) ---
# Always build the FULL arch (reduced_tail=False) so a pretrained=False model
# can load a checkpoint trained with pretrained=True.
def build_ssd(num_classes=6, pretrained=True):
    model = ssdlite320_mobilenet_v3_large(
        weights=None,
        weights_backbone=MobileNet_V3_Large_Weights.IMAGENET1K_V2,
        num_classes=num_classes)
    if not pretrained:
        import torch.nn as nn
        for m in model.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.03)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
    return model

# --- YOLO-format dataset for torchvision detectors ---
class DefectDataset(Dataset):
    def __init__(self, split):
        self.img_dir = DATA_DIR / 'images' / split
        self.lbl_dir = DATA_DIR / 'labels' / split
        self.images = sorted(p for p in self.img_dir.iterdir()
                             if p.suffix.lower() in IMAGE_EXTS)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        p = self.images[idx]
        img = Image.open(p).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        tensor = TF.to_tensor(img)
        boxes, labels = [], []
        lbl = self.lbl_dir / (p.stem + '.txt')
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = max(0.0, (cx - bw / 2) * IMG_SIZE)
                y1 = max(0.0, (cy - bh / 2) * IMG_SIZE)
                x2 = min(float(IMG_SIZE), (cx + bw / 2) * IMG_SIZE)
                y2 = min(float(IMG_SIZE), (cy + bh / 2) * IMG_SIZE)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls + 1)  # 0 = background
        if boxes:
            bt = torch.tensor(boxes, dtype=torch.float32)
            lt = torch.tensor(labels, dtype=torch.int64)
            area = (bt[:, 3] - bt[:, 1]) * (bt[:, 2] - bt[:, 0])
        else:
            bt = torch.zeros((0, 4), dtype=torch.float32)
            lt = torch.zeros(0, dtype=torch.int64)
            area = torch.zeros(0, dtype=torch.float32)
        target = {'boxes': bt, 'labels': lt, 'image_id': torch.tensor([idx]),
                  'area': area, 'iscrowd': torch.zeros(len(lt), dtype=torch.int64)}
        return tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

# drop_last=True: a final batch of size 1 breaks BatchNorm in training
ssd_train = DataLoader(DefectDataset('train'), batch_size=TORCH_BATCH, shuffle=True,
                       collate_fn=collate_fn, num_workers=WORKERS, drop_last=True)
ssd_val = DataLoader(DefectDataset('val'), batch_size=TORCH_BATCH, shuffle=False,
                     collate_fn=collate_fn, num_workers=WORKERS)
print('SSD loaders -> train batches: %d | val batches: %d' % (len(ssd_train), len(ssd_val)))

In [ ]:
# 2.3b SSDLite two-phase training loop
@torch.no_grad()
def ssd_evaluate(model, loader):
    model.eval()
    metric = MeanAveragePrecision(iou_type='bbox', class_metrics=True)
    for images, targets in tqdm(loader, desc='eval', leave=False):
        imgs = [i.to(DEVICE) for i in images]
        outs = model(imgs)
        metric.update(
            [{'boxes': o['boxes'].cpu(), 'scores': o['scores'].cpu(),
              'labels': o['labels'].cpu()} for o in outs],
            [{'boxes': t['boxes'].cpu(), 'labels': t['labels'].cpu()} for t in targets])
    torch.cuda.empty_cache()
    return metric.compute()

ssd_ver = 1
(RUNS_DIR / 'ssdlite').mkdir(parents=True, exist_ok=True)
while (RUNS_DIR / 'ssdlite' / ('best_v%d.pth' % ssd_ver)).exists():
    ssd_ver += 1
best_ckpt = RUNS_DIR / 'ssdlite' / ('best_v%d.pth' % ssd_ver)

ssd = build_ssd(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
ssd_params = sum(p.numel() for p in ssd.parameters()) / 1e6
print('SSDLite %.1fM params -> %s' % (ssd_params, best_ckpt.name))

# Phase 1: freeze backbone, train detection head only (epochs 1-5)
for p in ssd.backbone.parameters():
    p.requires_grad = False
opt = torch.optim.AdamW([p for p in ssd.parameters() if p.requires_grad],
                        lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_map = 0.0
t0 = time.time()
for epoch in range(1, TORCH_EPOCHS + 1):
    if epoch == 6:  # Phase 2: unfreeze backbone at 10x lower LR
        for p in ssd.backbone.parameters():
            p.requires_grad = True
        opt = torch.optim.AdamW(ssd.parameters(), lr=LR / 10, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TORCH_EPOCHS - 5)
    ssd.train()
    running = 0.0
    for images, targets in tqdm(ssd_train, desc='ep%d' % epoch, leave=False):
        imgs = [i.to(DEVICE) for i in images]
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        opt.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            loss = sum(ssd(imgs, tgts).values())
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running += loss.item()
    sched.step()
    metrics = ssd_evaluate(ssd, ssd_val)
    m50 = float(metrics['map_50'])
    print('  epoch %3d/%d loss=%.4f mAP@0.5=%.4f' % (epoch, TORCH_EPOCHS, running / len(ssd_train), m50))
    if m50 > best_map:
        best_map = m50
        torch.save({'epoch': epoch, 'model_state_dict': ssd.state_dict(),
                    'metrics': metrics}, best_ckpt)

ssd_min = (time.time() - t0) / 60
final = ssd_evaluate(ssd, ssd_val)
log_run('SSDLite', TORCH_EPOCHS, TORCH_BATCH, float(final['map_50']), float(final['map']),
        0.0, 0.0, ssd_min, ssd_params, str(best_ckpt))
print('SSDLite best mAP@0.5=%.4f (%.0f min)' % (best_map, ssd_min))

## Part 3 - Evaluation on the Test Set

Loads the best weights for each model and reports mAP@0.5, mAP@0.5:0.95,
precision/recall, speed, per-class AP, severity distribution, and qualitative samples.

In [ ]:
# 3.1 Load best weights for each model
import csv as _csv
import numpy as np

def best_yolo_weights(model_dir):
    versions = sorted([d for d in model_dir.iterdir() if d.is_dir() and d.name.startswith('v')],
                      key=lambda d: int(d.name[1:]))
    if not versions:
        raise FileNotFoundError('No runs in %s' % model_dir)
    best_ver, best = versions[-1], -1.0
    for v in versions:
        r = v / 'results.csv'
        if not r.exists() or r.stat().st_size == 0:
            continue
        try:
            rows = list(_csv.DictReader(open(r)))
            vb = max(float(x.get('metrics/mAP50(B)', 0)) for x in rows) if rows else 0.0
            if vb > best:
                best, best_ver = vb, v
        except Exception:
            continue
    return best_ver / 'weights' / 'best.pt'

models = {}
for nm, sub in [('YOLOv11n', 'yolo11n'), ('YOLOv8n', 'yolov8n')]:
    d = RUNS_DIR / sub
    if d.exists():
        try:
            w = best_yolo_weights(d)
            models[nm] = YOLO(str(w))
            print('Loaded %s from %s' % (nm, w))
        except FileNotFoundError as e:
            print(nm, ':', e)

ssd_dir = RUNS_DIR / 'ssdlite'
if ssd_dir.exists():
    ckpts = sorted(ssd_dir.glob('best_v*.pth'), key=lambda p: int(p.stem.split('_v')[1]))
    if ckpts:
        m = build_ssd(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
        m.load_state_dict(torch.load(str(ckpts[-1]), map_location=DEVICE)['model_state_dict'])
        m.eval()
        models['SSDLite'] = m
        print('Loaded SSDLite from %s' % ckpts[-1])

print('Models available:', list(models.keys()))

In [ ]:
# 3.2 Run test-set evaluation
test_loader = DataLoader(DefectDataset('test'), batch_size=4, shuffle=False,
                         collate_fn=collate_fn, num_workers=WORKERS)
CONF = 0.25

def yolo_speed(model, n=50):
    imgs = [p for p in (DATA_DIR / 'images' / 'test').iterdir()
            if p.suffix.lower() in IMAGE_EXTS][:n]
    ts = []
    for p in imgs:
        t0 = time.time()
        model.predict(str(p), verbose=False)
        ts.append((time.time() - t0) * 1000)
    return float(np.mean(ts)) if ts else 0.0

results = {}
for nm, model in models.items():
    print('Evaluating', nm)
    if nm in ('YOLOv11n', 'YOLOv8n'):
        val = model.val(data=DATA_YAML, split='test')
        results[nm] = {'mAP@0.5': round(float(val.box.map50), 4),
                       'mAP@0.5:0.95': round(float(val.box.map), 4),
                       'Precision': round(float(val.box.mp), 4),
                       'Recall': round(float(val.box.mr), 4),
                       'Speed (ms)': round(yolo_speed(model), 2)}
    else:
        model.eval()
        metric = MeanAveragePrecision(iou_type='bbox', class_metrics=True)
        times = []
        with torch.no_grad():
            for images, targets in tqdm(test_loader, desc='SSD test'):
                imgs = [i.to(DEVICE) for i in images]
                t0 = time.time()
                outs = model(imgs)
                times.append((time.time() - t0) / len(imgs) * 1000)
                preds = []
                for o in outs:
                    mask = o['scores'] >= CONF
                    preds.append({'boxes': o['boxes'][mask].cpu(),
                                  'scores': o['scores'][mask].cpu(),
                                  'labels': o['labels'][mask].cpu()})
                metric.update(preds, [{'boxes': t['boxes'].cpu(),
                                       'labels': t['labels'].cpu()} for t in targets])
        r = metric.compute()
        results[nm] = {'mAP@0.5': round(float(r['map_50']), 4),
                       'mAP@0.5:0.95': round(float(r['map']), 4),
                       'Precision': None, 'Recall': None,
                       'Speed (ms)': round(float(np.mean(times)), 2)}

In [ ]:
# 3.3 Comparison table + chart
comp = pd.DataFrame(results).T
comp.index.name = 'Model'
print(comp.to_string())
comp.to_csv(RUNS_DIR / 'model_comparison.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, metric in enumerate(['mAP@0.5', 'mAP@0.5:0.95']):
    names = [m for m in results if results[m][metric] is not None]
    vals = [results[m][metric] for m in names]
    axes[i].bar(names, vals, color=['#3498db', '#e74c3c', '#2ecc71'][:len(names)], edgecolor='black')
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    for j, v in enumerate(vals):
        axes[i].text(j, v + 0.01, '%.3f' % v, ha='center')
plt.suptitle('Model Comparison')
plt.tight_layout()
plt.savefig(RUNS_DIR / 'model_comparison.png', dpi=150)
plt.show()

In [ ]:
# 3.4 Per-class AP@0.5 (YOLO models)
for nm in ('YOLOv11n', 'YOLOv8n'):
    if nm not in models:
        continue
    vr = models[nm].val(data=DATA_YAML, split='test')
    per = dict(zip(CLASSES, vr.box.maps))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(list(per.keys()), list(per.values()), color=CLASS_COLORS, edgecolor='black')
    ax.set_title('%s - Per-Class AP@0.5' % nm)
    ax.set_ylim(0, 1)
    for i, (k, v) in enumerate(per.items()):
        ax.text(i, v + 0.01, '%.3f' % v, ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(RUNS_DIR / ('per_class_ap_%s.png' % nm.lower()), dpi=150)
    plt.show()

In [ ]:
# 3.5 Severity distribution on the test set (ground truth)
def severity_label(bw, bh):
    a = bw * bh * 100
    return 'Low' if a < 5 else ('Medium' if a <= 20 else 'High')

sev = {'Low': 0, 'Medium': 0, 'High': 0}
for lbl in (DATA_DIR / 'labels' / 'test').glob('*.txt'):
    for line in lbl.read_text().splitlines():
        parts = line.split()
        if len(parts) == 5:
            sev[severity_label(float(parts[3]), float(parts[4]))] += 1
print('Severity (test GT):', sev)

fig, ax = plt.subplots(figsize=(5, 4))
ax.pie(list(sev.values()), labels=list(sev.keys()), autopct='%1.1f%%',
       colors=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_title('Ground-Truth Severity Distribution (Test)')
plt.tight_layout()
plt.savefig(RUNS_DIR / 'severity_distribution.png', dpi=150)
plt.show()

In [ ]:
# 3.6 Qualitative predictions (YOLOv11n) with severity tags
if 'YOLOv11n' in models:
    test_imgs = sorted(p for p in (DATA_DIR / 'images' / 'test').iterdir()
                       if p.suffix.lower() in IMAGE_EXTS)[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, p in enumerate(test_imgs):
        res = models['YOLOv11n'].predict(str(p), conf=CONF, verbose=False)[0]
        img = Image.open(p).convert('RGB')
        axes[i].imshow(img)
        w, h = img.size
        for box in res.boxes:
            cls = int(box.cls.item())
            conf = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            s = severity_label((x2 - x1) / w, (y2 - y1) / h)
            axes[i].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, lw=2,
                              edgecolor=CLASS_COLORS[cls], facecolor='none'))
            axes[i].text(x1, y1 - 5, '%s %.2f [%s]' % (CLASSES[cls], conf, s),
                         color=CLASS_COLORS[cls], fontsize=7, fontweight='bold')
        axes[i].axis('off')
        axes[i].set_title(p.name, fontsize=8)
    plt.suptitle('YOLOv11n Predictions with Severity')
    plt.tight_layout()
    plt.savefig(RUNS_DIR / 'prediction_samples.png', dpi=120)
    plt.show()
else:
    print('YOLOv11n not available - skipping qualitative samples.')

## Part 4 - Save Results back to Google Drive

Colab runtimes are ephemeral - anything not saved to Drive is lost on disconnect.
This zips the whole `runs/` folder (weights + charts + logs) to
`MyDrive/COS40007/runs_results.zip`. Download it and unzip into your local
project's `runs/` to use the weights in the Streamlit UI.

In [ ]:
if IN_COLAB:
    dst = Path('/content/drive/MyDrive/COS40007')
    dst.mkdir(parents=True, exist_ok=True)
    zip_base = WORK_DIR / 'runs_results'
    shutil.make_archive(str(zip_base), 'zip', root_dir=str(RUNS_DIR.parent), base_dir='runs')
    shutil.copy(str(zip_base) + '.zip', dst / 'runs_results.zip')
    print('Saved:', dst / 'runs_results.zip')
    print('Download it and unzip into your local project so runs/ has the new weights.')
else:
    print('Local run - results already in', RUNS_DIR)